In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Case Study: Building a Regulatory Compliance RAG System

## Implementation Notebook

*Meridian Compliance Technologies — From BM25 Baseline to Production-Grade RAG*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/context-engineering/rag-systems/practice/0/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


## 1. Environment Setup and Data Acquisition

In this notebook, you will build a retrieval-augmented generation (RAG) system for regulatory compliance question answering. The scenario: Meridian Compliance Technologies needs to replace their keyword-based search with a system that understands the *meaning* of compliance questions and generates faithful, citation-backed answers.

We use the **CUAD (Contract Understanding Atticus Dataset)** — 510 real commercial legal contracts with 13,101 expert annotations across 41 legal clause types.

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu rank_bm25 datasets transformers torch matplotlib seaborn pandas numpy scikit-learn tqdm

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
import torch
torch.manual_seed(42)

print("Environment ready.")

In [ ]:
from datasets import load_dataset

# Load CUAD dataset
dataset = load_dataset("cuad", trust_remote_code=True)
train_data = dataset["test"]  # CUAD uses 'test' split for the full annotated set

print(f"Number of contracts: {len(train_data)}")
print(f"Features: {train_data.features}")
print(f"\nSample entry keys: {list(train_data[0].keys())}")

### 1.1 Understanding the CUAD Structure

Each entry in CUAD contains a contract's full text and annotations for 41 clause types. Let us parse this into a structured format.

In [ ]:
# Parse CUAD into contracts with annotations
contracts = []
clause_types = [
    "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Most Favored Nation", "Non-Compete",
    "Exclusivity", "No-Solicit Of Customers", "Competitive Restriction Exception",
    "No-Solicit Of Employees", "Non-Disparagement", "Termination For Convenience",
    "Rofr/Rofo/Rofn", "Change Of Control", "Anti-Assignment",
    "Revenue/Profit Sharing", "Price Restrictions", "Minimum Commitment",
    "Volume Restriction", "Ip Ownership Assignment", "Joint Ip Ownership",
    "License Grant", "Non-Transferable License", "Affiliate License-Licensor",
    "Affiliate License-Licensee", "Unlimited/All-You-Can-Eat-License",
    "Irrevocable Or Perpetual License", "Source Code Escrow",
    "Post-Termination Services", "Audit Rights", "Uncapped Liability",
    "Cap On Liability", "Liquidated Damages", "Warranty Duration",
    "Insurance", "Covenant Not To Sue", "Third Party Beneficiary"
]

for i, entry in enumerate(train_data):
    context = entry["context"]
    annotations = {}
    for clause in clause_types:
        key = clause.lower().replace(" ", "_").replace("/", "_").replace("-", "_")
        answers = entry.get("answers", {})
        if "text" in answers and len(answers["text"]) > 0:
            annotations[clause] = answers["text"]

    contracts.append({
        "id": i,
        "title": entry.get("title", f"Contract_{i}"),
        "text": context,
        "annotations": annotations
    })

print(f"Parsed {len(contracts)} contracts")
print(f"Sample contract length: {len(contracts[0]['text'].split())} words")

### 1.2 Generate Q&A Pairs from Annotations

To evaluate our RAG system, we need (question, answer, evidence_passage) triples. We generate these from CUAD's clause annotations.

In [ ]:
# Example Q&A generation for a few clause types
clause_questions = {
    "Termination For Convenience": "What are the termination for convenience provisions?",
    "Governing Law": "Which governing law applies to this contract?",
    "Non-Compete": "What non-compete restrictions are specified?",
    "Expiration Date": "When does this agreement expire?",
    "License Grant": "What license rights are granted?",
    "Cap On Liability": "What are the liability caps?",
    "Insurance": "What insurance requirements are specified?",
    "Audit Rights": "What audit rights are included?",
    "Ip Ownership Assignment": "How is intellectual property ownership assigned?",
    "Renewal Term": "What are the renewal terms?"
}

print("Clause type -> Question mapping:")
for clause, question in list(clause_questions.items())[:5]:
    print(f"  {clause}: '{question}'")

In [ ]:
def generate_qa_pairs(contracts, clause_questions, max_pairs=500):
    """
    Generate (question, answer, evidence, contract_id) tuples
    from CUAD annotations.

    For each contract and each annotated clause type:
    1. The question comes from our clause_questions mapping
    2. The answer is the annotated text span
    3. The evidence is the surrounding context (±200 chars)
    """
    # ============ TODO ============
    # Implement this function:
    # Step 1: Iterate over contracts
    # Step 2: For each contract, check which clause types have annotations
    # Step 3: For annotated clauses, create a QA pair:
    #         - question: from clause_questions dict
    #         - answer: the annotation text
    #         - evidence: extract ±200 chars around the annotation in the contract
    #         - contract_id: the contract index
    # Step 4: Return list of QA pair dicts
    # Hint: Use str.find() to locate the annotation in the contract text
    # ==============================

    qa_pairs = []  # YOUR CODE HERE

    return qa_pairs[:max_pairs]

# Generate QA pairs
qa_pairs = generate_qa_pairs(contracts, clause_questions)
print(f"Generated {len(qa_pairs)} QA pairs")
if qa_pairs:
    print(f"\nSample QA pair:")
    print(f"  Q: {qa_pairs[0].get('question', 'N/A')}")
    print(f"  A: {str(qa_pairs[0].get('answer', 'N/A'))[:100]}...")

## 2. Exploratory Data Analysis

Before building the RAG pipeline, let us understand our data. The distributions we discover here will directly inform our chunking and retrieval strategy.

In [ ]:
# Document length distribution
doc_lengths = [len(c["text"].split()) for c in contracts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(doc_lengths, bins=50, color="#2563eb", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Document Length (words)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Contract Lengths")
axes[0].axvline(np.median(doc_lengths), color="red", linestyle="--",
                label=f"Median: {np.median(doc_lengths):.0f}")
axes[0].legend()

axes[1].boxplot(doc_lengths, vert=True)
axes[1].set_ylabel("Document Length (words)")
axes[1].set_title("Contract Length Boxplot")

plt.tight_layout()
plt.show()

print(f"Length stats: min={min(doc_lengths)}, max={max(doc_lengths)}, "
      f"median={np.median(doc_lengths):.0f}, mean={np.mean(doc_lengths):.0f}")

In [ ]:
# Clause type frequency across contracts
clause_counts = Counter()
for c in contracts:
    for clause_type in c["annotations"]:
        clause_counts[clause_type] += 1

# Plot top 20 clause types
top_clauses = clause_counts.most_common(20)
clause_names = [c[0] for c in top_clauses]
clause_freqs = [c[1] for c in top_clauses]

plt.figure(figsize=(12, 6))
plt.barh(range(len(clause_names)), clause_freqs, color="#7c3aed", alpha=0.8)
plt.yticks(range(len(clause_names)), clause_names, fontsize=9)
plt.xlabel("Number of Contracts")
plt.title("Most Common Clause Types in CUAD")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ============ TODO ============
# Analyze annotation span lengths by clause type:
# 1. For each clause type, compute the average character length of annotations
# 2. Create a grouped bar chart showing mean and std of span lengths
#    for the top 10 most common clause types
# 3. Answer: Which clause types have the highest variance in span length?
#    What does this tell us about our chunking strategy?
# Hint: Long, variable-length clauses may need larger chunks or
#       semantic chunking to avoid boundary splits.
# ==============================

# YOUR CODE HERE

## 3. Baseline: BM25 Keyword Search

Our baseline mirrors Meridian's current Elasticsearch approach. BM25 (Best Matching 25) scores documents based on term frequency and inverse document frequency — it rewards documents that contain rare query terms frequently.

The BM25 score for a document $d$ given query $q$ is:

$$\text{BM25}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{avgdl})}$$

where $f(t,d)$ is the term frequency of $t$ in $d$, $|d|$ is the document length, $avgdl$ is the average document length, $k_1 = 1.5$ controls term frequency saturation, and $b = 0.75$ controls length normalization.

In [ ]:
from rank_bm25 import BM25Okapi
import re

def chunk_contract(text, chunk_size=500, overlap=100):
    """Split contract text into overlapping word chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])
        chunks.append(chunk_text)
        if end >= len(words):
            break
        start = end - overlap
    return chunks

# Chunk all contracts
all_chunks = []
chunk_metadata = []  # Track which contract each chunk belongs to

for contract in tqdm(contracts, desc="Chunking contracts"):
    chunks = chunk_contract(contract["text"])
    for j, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "contract_id": contract["id"],
            "chunk_index": j,
            "title": contract["title"]
        })

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunks per contract: {len(all_chunks)/len(contracts):.1f}")

In [ ]:
# Build BM25 index
tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, top_k=10):
    """Search using BM25 and return top-k chunks with scores."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "chunk_index": idx,
            "score": scores[idx],
            "text": all_chunks[idx][:200] + "...",
            "metadata": chunk_metadata[idx]
        })
    return results

# Test BM25 search
sample_query = "What are the termination provisions?"
results = bm25_search(sample_query, top_k=5)
print(f"Query: '{sample_query}'\n")
for i, r in enumerate(results):
    print(f"  [{i+1}] Score: {r['score']:.2f} | Contract: {r['metadata']['title']}")
    print(f"      {r['text'][:120]}...")
    print()

In [ ]:
def evaluate_retrieval(qa_pairs, search_fn, top_k=10):
    """
    Evaluate retrieval quality using Recall@k and MRR.

    Recall@k: fraction of queries where the relevant chunk
              appears in the top-k results.
    MRR: Mean Reciprocal Rank — average of 1/rank for the
         first relevant result.
    """
    recalls = []
    mrrs = []

    for qa in tqdm(qa_pairs[:100], desc="Evaluating"):
        query = qa["question"]
        evidence = qa.get("evidence", qa.get("answer", ""))
        results = search_fn(query, top_k=top_k)

        # Check if any retrieved chunk contains the evidence
        found = False
        for rank, r in enumerate(results, 1):
            chunk_text = all_chunks[r["chunk_index"]]
            # Fuzzy match: check if significant overlap exists
            evidence_words = set(str(evidence).lower().split()[:20])
            chunk_words = set(chunk_text.lower().split())
            overlap = len(evidence_words & chunk_words) / max(len(evidence_words), 1)

            if overlap > 0.5:
                recalls.append(1.0)
                mrrs.append(1.0 / rank)
                found = True
                break

        if not found:
            recalls.append(0.0)
            mrrs.append(0.0)

    recall_at_k = np.mean(recalls)
    mrr = np.mean(mrrs)
    return recall_at_k, mrr

# Evaluate BM25 baseline
if qa_pairs:
    bm25_recall, bm25_mrr = evaluate_retrieval(qa_pairs, bm25_search, top_k=10)
    print(f"BM25 Baseline:")
    print(f"  Recall@10: {bm25_recall:.3f}")
    print(f"  MRR:       {bm25_mrr:.3f}")
else:
    print("No QA pairs generated. Complete TODO in Section 1.2 first.")
    bm25_recall, bm25_mrr = 0.0, 0.0

## 4. Building the RAG Pipeline — Component by Component

### 4.1 Dense Embeddings

BM25 matches keywords, but compliance questions often use different words than the regulatory text. "Can we do X?" might match a regulation that says "Entity shall not engage in X" — same meaning, zero keyword overlap.

Dense embeddings solve this by mapping text into a vector space where semantically similar texts are close together, regardless of exact wording.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = 384

print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {embedding_dim}")
print(f"Max sequence length: {embed_model.max_seq_length}")

# Test embedding
test_texts = [
    "What are the termination provisions?",
    "Either party may terminate this agreement upon 30 days written notice.",
    "The weather today is sunny and warm."
]
test_embeddings = embed_model.encode(test_texts, normalize_embeddings=True)

# Compute cosine similarities
from numpy.linalg import norm
for i in range(len(test_texts)):
    for j in range(i+1, len(test_texts)):
        sim = np.dot(test_embeddings[i], test_embeddings[j])
        print(f"Similarity('{test_texts[i][:40]}...', '{test_texts[j][:40]}...'): {sim:.3f}")

In [ ]:
# Embed all chunks (this takes a few minutes)
print(f"Embedding {len(all_chunks)} chunks...")
chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"Embeddings shape: {chunk_embeddings.shape}")

### 4.2 Vector Index with FAISS

With 1.4 million vectors in production, brute-force search is too slow. FAISS provides approximate nearest neighbor (ANN) algorithms that trade a small amount of recall for dramatic speedups.

We use IVF (Inverted File Index): partition the vector space into clusters, then at query time only search the closest clusters.

In [ ]:
import faiss

# Build FAISS index
# For our dataset size, a flat index is fine, but we'll also build
# an IVF index to demonstrate the production approach

# Flat index (exact search - baseline)
flat_index = faiss.IndexFlatIP(embedding_dim)  # Inner product = cosine sim for normalized vectors
flat_index.add(chunk_embeddings.astype(np.float32))
print(f"Flat index: {flat_index.ntotal} vectors")

# IVF index (approximate search - production)
n_clusters = min(100, len(all_chunks) // 10)
quantizer = faiss.IndexFlatIP(embedding_dim)
ivf_index = faiss.IndexIVFFlat(quantizer, embedding_dim, n_clusters, faiss.METRIC_INNER_PRODUCT)
ivf_index.train(chunk_embeddings.astype(np.float32))
ivf_index.add(chunk_embeddings.astype(np.float32))
ivf_index.nprobe = 10  # Search 10 nearest clusters
print(f"IVF index: {ivf_index.ntotal} vectors, {n_clusters} clusters, nprobe={ivf_index.nprobe}")

In [ ]:
def dense_search(query, index, top_k=10):
    """Search using dense embeddings + FAISS."""
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding.astype(np.float32), top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            "chunk_index": int(idx),
            "score": float(score),
            "text": all_chunks[idx][:200] + "...",
            "metadata": chunk_metadata[idx]
        })
    return results

# Test dense search
results = dense_search("What are the termination provisions?", flat_index, top_k=5)
print(f"Dense search results:\n")
for i, r in enumerate(results):
    print(f"  [{i+1}] Score: {r['score']:.3f} | Contract: {r['metadata']['title']}")
    print(f"      {r['text'][:120]}...")
    print()

### 4.3 Hybrid Retrieval with Reciprocal Rank Fusion

Neither dense nor sparse retrieval is strictly better. Dense retrieval captures semantics ("termination" matches "cancellation") but can miss exact identifiers ("Section 12(b)"). Sparse retrieval catches identifiers but misses paraphrases. Hybrid retrieval combines both.

Reciprocal Rank Fusion (RRF) merges ranked lists by converting ranks to scores:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k_0 + \text{rank}_r(d)}$$

The key insight: RRF is rank-based, not score-based. This means we do not need to calibrate the vastly different score distributions of BM25 and cosine similarity.

In [ ]:
def hybrid_search(query, top_k=10, k0=60, dense_weight=1.0, sparse_weight=1.0):
    """
    Hybrid search combining dense and BM25 retrieval using RRF.
    """
    # ============ TODO ============
    # Implement Reciprocal Rank Fusion:
    # Step 1: Get top-50 results from dense search (use flat_index)
    # Step 2: Get top-50 results from BM25 search
    # Step 3: For each unique chunk that appears in either result set:
    #         - Compute RRF score = dense_weight/(k0 + dense_rank) + sparse_weight/(k0 + sparse_rank)
    #         - If a chunk only appears in one list, use rank = 1000 for the missing list
    # Step 4: Sort by RRF score descending
    # Step 5: Return top_k results
    # ==============================

    results = []  # YOUR CODE HERE

    return results[:top_k]

# Test hybrid search
results = hybrid_search("What are the termination provisions?", top_k=5)
if results:
    print("Hybrid search results:")
    for i, r in enumerate(results):
        print(f"  [{i+1}] RRF Score: {r.get('score', 0):.4f}")
else:
    print("Complete the TODO above to see hybrid search results.")

In [ ]:
# Verification: compare retrieval methods
print("=" * 60)
print("Retrieval Method Comparison")
print("=" * 60)

test_queries = [
    "What are the termination provisions?",
    "Which governing law applies?",
    "What non-compete restrictions exist?",
    "What are the liability caps?",
    "How is intellectual property assigned?"
]

for query in test_queries:
    bm25_results = bm25_search(query, top_k=5)
    dense_results = dense_search(query, flat_index, top_k=5)

    bm25_chunks = set(r["chunk_index"] for r in bm25_results)
    dense_chunks = set(r["chunk_index"] for r in dense_results)
    overlap = len(bm25_chunks & dense_chunks)

    print(f"\nQuery: '{query}'")
    print(f"  BM25 top-5 chunks:  {sorted(bm25_chunks)}")
    print(f"  Dense top-5 chunks: {sorted(dense_chunks)}")
    print(f"  Overlap: {overlap}/5 ({overlap/5*100:.0f}%)")

### 4.4 Cross-Encoder Reranking

The hybrid retriever is fast but uses bi-encoder similarity — query and document are encoded independently. A cross-encoder processes the (query, document) pair jointly, allowing deep interaction between them. This is much more accurate but too slow for first-stage retrieval.

We use it as a second stage: retrieve 50 candidates with hybrid search, then rerank with the cross-encoder to get the top 5.

In [ ]:
from sentence_transformers import CrossEncoder

# Load cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

def rerank_results(query, candidates, top_k=5):
    """
    Rerank candidate chunks using a cross-encoder.

    The cross-encoder scores each (query, chunk) pair jointly,
    capturing fine-grained relevance that bi-encoders miss.
    """
    if not candidates:
        return []

    # Prepare pairs for the cross-encoder
    pairs = [(query, all_chunks[c["chunk_index"]]) for c in candidates]

    # Score all pairs
    scores = reranker.predict(pairs)

    # Attach scores and sort
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

# Test reranking
query = "What are the termination provisions?"
# Get 50 candidates from dense search (or hybrid if implemented)
candidates = dense_search(query, flat_index, top_k=50)
reranked = rerank_results(query, candidates, top_k=5)

print(f"Query: '{query}'\n")
print("Before reranking (top 5 by dense score):")
for i, r in enumerate(candidates[:5]):
    print(f"  [{i+1}] Dense: {r['score']:.3f} | {r['text'][:80]}...")

print("\nAfter reranking (top 5 by cross-encoder):")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] Rerank: {r['rerank_score']:.3f} | {r['text'][:80]}...")

### 4.5 Prompt Construction and Generation

The final piece: construct a prompt that instructs the LLM to answer based ONLY on the retrieved context, with citations.

Since this notebook runs in Colab without an LLM API key, we will construct the prompt and demonstrate the format. In production, this would be sent to Claude or Llama.

In [ ]:
def construct_rag_prompt(query, retrieved_chunks, max_context_tokens=2000):
    """
    Build the RAG prompt with retrieved context and citation instructions.
    """
    system_prompt = """You are a regulatory compliance assistant for Meridian Compliance Technologies.
Answer the user's question based ONLY on the provided context documents.

Rules:
1. Every factual claim must cite its source using [Source N] notation.
2. If the context does not contain enough information to answer, say "I could not find sufficient regulatory guidance for this question."
3. Never generate information not present in the context.
4. Be precise and quote relevant passages when helpful."""

    # Format context with source numbers
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        chunk_text = all_chunks[chunk["chunk_index"]]
        # Truncate if needed
        words = chunk_text.split()[:max_context_tokens // len(retrieved_chunks)]
        context_parts.append(f"[Source {i}] (Contract: {chunk['metadata']['title']}, "
                           f"Chunk {chunk['metadata']['chunk_index']})\n"
                           f"{' '.join(words)}")

    context = "\n\n---\n\n".join(context_parts)

    user_prompt = f"""Context Documents:

{context}

---

Question: {query}

Answer (cite sources using [Source N]):"""

    return system_prompt, user_prompt

# Construct a sample prompt
query = "What are the termination provisions?"
top_chunks = reranked if reranked else dense_search(query, flat_index, top_k=5)
system_prompt, user_prompt = construct_rag_prompt(query, top_chunks)

print("=== SYSTEM PROMPT ===")
print(system_prompt)
print("\n=== USER PROMPT (first 500 chars) ===")
print(user_prompt[:500] + "...\n")
print(f"Total prompt length: ~{len(system_prompt.split()) + len(user_prompt.split())} words")

## 5. Evaluation

### 5.1 Retrieval Quality Comparison

In [ ]:
# Compare all retrieval methods
print("Evaluating retrieval methods (this may take a few minutes)...\n")

methods = {
    "BM25": lambda q, k: bm25_search(q, top_k=k),
    "Dense (FAISS)": lambda q, k: dense_search(q, flat_index, top_k=k),
}

# Only add reranked if hybrid_search is implemented
try:
    test = hybrid_search("test", top_k=1)
    if test:
        methods["Hybrid (RRF)"] = lambda q, k: hybrid_search(q, top_k=k)
        methods["Hybrid + Rerank"] = lambda q, k: rerank_results(q, hybrid_search(q, top_k=50), top_k=k)
except:
    pass

results_table = []
if qa_pairs:
    for name, search_fn in methods.items():
        recall, mrr = evaluate_retrieval(qa_pairs, search_fn, top_k=10)
        results_table.append({"Method": name, "Recall@10": recall, "MRR": mrr})
        print(f"{name:25s} | Recall@10: {recall:.3f} | MRR: {mrr:.3f}")
else:
    print("No QA pairs. Complete TODO in Section 1.2 first.")

In [ ]:
# Visualize results
if results_table:
    df_results = pd.DataFrame(results_table)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    colors = ["#94a3b8", "#2563eb", "#7c3aed", "#059669"]

    axes[0].bar(range(len(df_results)), df_results["Recall@10"],
                color=colors[:len(df_results)], alpha=0.85)
    axes[0].set_xticks(range(len(df_results)))
    axes[0].set_xticklabels(df_results["Method"], rotation=15, ha="right")
    axes[0].set_ylabel("Recall@10")
    axes[0].set_title("Retrieval Recall@10 by Method")
    axes[0].set_ylim(0, 1)
    axes[0].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="Target: 0.85")
    axes[0].legend()

    axes[1].bar(range(len(df_results)), df_results["MRR"],
                color=colors[:len(df_results)], alpha=0.85)
    axes[1].set_xticks(range(len(df_results)))
    axes[1].set_xticklabels(df_results["Method"], rotation=15, ha="right")
    axes[1].set_ylabel("MRR")
    axes[1].set_title("Mean Reciprocal Rank by Method")
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.show()

### 5.2 Chunk Size Ablation

How does chunk size affect retrieval quality? Larger chunks carry more context but may dilute the signal. Smaller chunks are more precise but may split relevant information.

In [ ]:
# ============ TODO ============
# Chunk size ablation study:
# 1. Re-chunk all contracts with chunk_size in [256, 512, 1024] (keep overlap=100)
# 2. For each chunk size:
#    a. Build BM25 index on the new chunks
#    b. Embed the chunks with the sentence-transformer
#    c. Build a FAISS flat index
#    d. Evaluate BM25 and Dense Recall@10 on qa_pairs[:50]
# 3. Plot chunk_size (x-axis) vs Recall@10 (y-axis) for both methods
# 4. Answer: What is the optimal chunk size? Why?
# ==============================

# YOUR CODE HERE

## 6. Error Analysis

Understanding where the system fails is as important as measuring where it succeeds.

In [ ]:
# Identify failure cases from BM25 baseline
if qa_pairs:
    failures = []
    successes = []

    for qa in qa_pairs[:100]:
        query = qa["question"]
        evidence = qa.get("evidence", qa.get("answer", ""))
        results = bm25_search(query, top_k=10)

        found = False
        for r in results:
            chunk_text = all_chunks[r["chunk_index"]]
            evidence_words = set(str(evidence).lower().split()[:20])
            chunk_words = set(chunk_text.lower().split())
            overlap = len(evidence_words & chunk_words) / max(len(evidence_words), 1)
            if overlap > 0.5:
                found = True
                break

        if not found:
            failures.append(qa)
        else:
            successes.append(qa)

    print(f"Successes: {len(successes)}, Failures: {len(failures)}")
    print(f"\nSample failure cases:")
    for f in failures[:3]:
        print(f"  Q: {f['question']}")
        print(f"  Expected: {str(f.get('answer', 'N/A'))[:100]}...")
        print()
else:
    print("No QA pairs. Complete TODO in Section 1.2 first.")

In [ ]:
# ============ TODO ============
# Categorize the failure cases:
# 1. Examine the first 10 failure cases
# 2. For each, determine the failure category:
#    a. "semantic_miss" - query uses different words than the document
#    b. "boundary_split" - relevant info is split across chunk boundaries
#    c. "metadata_dependent" - query requires metadata (date, source) not in chunk text
#    d. "long_range" - answer requires info from multiple distant sections
# 3. Count the frequency of each category
# 4. Propose one specific mitigation for the most common failure mode
# ==============================

# YOUR CODE HERE

## 7. Scalability and Deployment

### 7.1 Latency Profiling

In [ ]:
import time

def benchmark_latency(search_fn, queries, n_runs=3):
    """Measure P50, P95, P99 latency for a search function."""
    latencies = []
    for _ in range(n_runs):
        for q in queries:
            start = time.perf_counter()
            search_fn(q, 10)
            elapsed = (time.perf_counter() - start) * 1000  # ms
            latencies.append(elapsed)

    latencies = sorted(latencies)
    n = len(latencies)
    return {
        "P50": latencies[n // 2],
        "P95": latencies[int(n * 0.95)],
        "P99": latencies[int(n * 0.99)],
        "mean": np.mean(latencies)
    }

test_queries = [
    "What are the termination provisions?",
    "Which governing law applies?",
    "What non-compete restrictions exist?",
    "What are the liability caps?",
    "How is intellectual property assigned?",
    "What audit rights are included?",
    "What insurance requirements exist?",
    "What are the renewal terms?",
    "What confidentiality obligations apply?",
    "What are the payment terms?"
]

print("Latency Profiling (10 queries, 3 runs each):\n")

bm25_latency = benchmark_latency(bm25_search, test_queries)
print(f"BM25:  P50={bm25_latency['P50']:.1f}ms, P95={bm25_latency['P95']:.1f}ms, "
      f"P99={bm25_latency['P99']:.1f}ms")

dense_latency = benchmark_latency(
    lambda q, k: dense_search(q, flat_index, top_k=k), test_queries)
print(f"Dense: P50={dense_latency['P50']:.1f}ms, P95={dense_latency['P95']:.1f}ms, "
      f"P99={dense_latency['P99']:.1f}ms")

ivf_latency = benchmark_latency(
    lambda q, k: dense_search(q, ivf_index, top_k=k), test_queries)
print(f"IVF:   P50={ivf_latency['P50']:.1f}ms, P95={ivf_latency['P95']:.1f}ms, "
      f"P99={ivf_latency['P99']:.1f}ms")

In [ ]:
# ============ TODO ============
# Scalability estimation:
# 1. Our current corpus has len(all_chunks) chunks.
#    Meridian's production corpus will have ~1.4M chunks.
#    Estimate the memory required for 1.4M 384-dim float32 vectors.
# 2. Compare index build time for Flat vs IVF on current data.
#    Extrapolate: what would the build times be for 1.4M vectors?
# 3. Compare query latency for Flat vs IVF as corpus grows.
#    At what corpus size does IVF become necessary (Flat > 100ms)?
# ==============================

# YOUR CODE HERE

## 8. Putting It All Together: Full RAG Pipeline

In [ ]:
def full_rag_pipeline(query, top_k=5, use_reranking=True):
    """
    Complete RAG pipeline: query -> retrieve -> rerank -> generate prompt.

    Returns the constructed prompt and retrieved sources.
    """
    print(f"Query: {query}\n")

    # Step 1: Retrieve candidates
    print("[1/4] Retrieving candidates...")
    # Try hybrid search first, fall back to dense
    try:
        candidates = hybrid_search(query, top_k=50)
        if not candidates:
            raise ValueError("Empty results")
        retrieval_method = "Hybrid (RRF)"
    except:
        candidates = dense_search(query, flat_index, top_k=50)
        retrieval_method = "Dense (FAISS)"
    print(f"      {retrieval_method}: {len(candidates)} candidates")

    # Step 2: Rerank
    if use_reranking and candidates:
        print("[2/4] Reranking with cross-encoder...")
        top_chunks = rerank_results(query, candidates, top_k=top_k)
        print(f"      Reranked to top-{top_k}")
    else:
        top_chunks = candidates[:top_k]
        print(f"[2/4] Skipping reranking, using top-{top_k}")

    # Step 3: Construct prompt
    print("[3/4] Constructing RAG prompt...")
    system_prompt, user_prompt = construct_rag_prompt(query, top_chunks)
    total_words = len(system_prompt.split()) + len(user_prompt.split())
    print(f"      Prompt: ~{total_words} words")

    # Step 4: (Simulated) Generation
    print("[4/4] Ready for LLM generation")
    print(f"      In production, this prompt would be sent to Claude/Llama.\n")

    # Display retrieved sources
    print("=" * 50)
    print("Retrieved Sources:")
    print("=" * 50)
    for i, chunk in enumerate(top_chunks, 1):
        score_key = "rerank_score" if "rerank_score" in chunk else "score"
        print(f"\n[Source {i}] Score: {chunk[score_key]:.3f}")
        print(f"  Contract: {chunk['metadata']['title']}")
        print(f"  Excerpt: {all_chunks[chunk['chunk_index']][:150]}...")

    return system_prompt, user_prompt, top_chunks

# Run the full pipeline
system_prompt, user_prompt, sources = full_rag_pipeline(
    "What are the termination for convenience provisions in this contract?"
)

In [ ]:
# Run on multiple queries
print("\n" + "=" * 60)
print("FULL RAG PIPELINE — BATCH EVALUATION")
print("=" * 60)

test_queries = [
    "What governing law applies to this agreement?",
    "What are the non-compete restrictions?",
    "How is intellectual property ownership assigned?",
    "What are the liability limitations?",
    "What audit rights does the licensor have?"
]

for query in test_queries:
    print(f"\n{'─' * 60}")
    _, _, sources = full_rag_pipeline(query, top_k=3, use_reranking=True)

## 9. Ethical and Regulatory Analysis

### Bias and Fairness Considerations

RAG systems in the legal domain face unique ethical challenges:

1. **Historical bias in legal language**: Older contracts may use gendered language ("he/his" for any party), culturally specific assumptions, or outdated legal frameworks. The embedding model trained on this data may encode these biases.

2. **Jurisdictional fairness**: If the training corpus is dominated by US contracts (as CUAD is), the system may perform poorly on contracts governed by UK, EU, or Asian legal frameworks.

3. **Accountability gap**: When a RAG system provides incorrect compliance guidance, the chain of responsibility is unclear. The LLM provider disclaim liability in their terms of service. The tool vendor (Meridian) has professional liability but may argue the human user should have verified. The user may argue they relied on the system as an expert tool.

In [ ]:
# ============ TODO ============
# Write a brief ethical impact assessment (in a print statement
# or markdown cell) addressing:
# 1. Who should be held accountable when a RAG compliance system
#    provides incorrect guidance that a client acts upon?
# 2. What minimum safeguards should Meridian implement to ensure
#    the system does not amplify biases in historical legal text?
# 3. How should the system handle queries about regulations it
#    was not trained on (e.g., a new jurisdiction)?
# Your assessment should be ~300 words.
# ==============================

# YOUR CODE HERE

## Summary

In this notebook, you built a regulatory compliance RAG system from scratch:

1. **Data**: Loaded and parsed the CUAD legal contract dataset
2. **Baseline**: Implemented BM25 keyword search and measured its limitations
3. **Dense Retrieval**: Embedded chunks with sentence-transformers and indexed with FAISS
4. **Hybrid Search**: Combined dense and sparse retrieval using Reciprocal Rank Fusion
5. **Reranking**: Applied a cross-encoder to refine the top results
6. **Prompt Construction**: Built citation-aware RAG prompts for LLM generation
7. **Evaluation**: Compared retrieval methods on Recall@10 and MRR
8. **Error Analysis**: Categorized failure modes and proposed mitigations

For the full production system design — including API design, serving infrastructure, latency budgets, monitoring, and A/B testing — see **Section 4** of the case study document.

Key takeaway: RAG is not a single algorithm but a *pipeline* where every component — chunking, embedding, retrieval, reranking, prompting — affects the final answer quality. The best RAG systems carefully optimize each stage while measuring end-to-end faithfulness.